# Exploring Prompting

In this notebook, we'll explore different prompting techniques with a focus on RAG systems.

## Recommended Hardware

This notebook can run on the following hardware or remote resources

✅ AMD Instinct™ Accelerators  
✅ AMD Radeon™ RX/PRO Graphics Cards  
✅ AMD EPYC™ Processors  
✅ AMD Ryzen™ (AI) Processors  

[![Open in AMD Developer Cloud](https://img.shields.io/badge/Open_in_AMD_Developer_Cloud-000000?logo=amd&logoSize=auto)](https://amd-ai-academy.com/github/AMDResearch/aup-ai-tutorials/blob/main/rag/02.rag-prompts.ipynb)  


## Software Environment

Install ROCm on your system

| Linux | Windows |
|-------|---------|
| [Install PyTorch](https://rocm.docs.amd.com/projects/install-on-linux/en/latest/install/quick-start.html) | [PyTorch on Windows](https://rocm.docs.amd.com/projects/radeon-ryzen/en/latest/docs/install/installrad/windows/install-pytorch.html)|
| [Install Docker container](https://amdresearch.github.io/aup-ai-tutorials//env/env-gpu.html) | |

## Goals

- Understand the difference between `system` and `user` prompts
- Build and reuse prompt templates with `ChatPromptTemplate`
- Create a simple RAG prompt using `{context}` and `{question}`

### Install Dependencies

Install the package dependencies needed for this notebook.

First, get the `aup_config.py` script locally with the package dependencies.

In [ ]:
!wget https://raw.githubusercontent.com/AMDResearch/aup-ai-tutorials/refs/heads/main/rag/aup_config.py

Install the dependencies. This step may take a few minutes and only needs to be done once.

In [ ]:
from aup_config import aup_setup
aup_setup()

## Prompt Engineering Basics for RAG

### Import Libraries

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate

### Manually Defining a Prompt

One of the simplest ways to define a prompt is to use a list of dictionaries, where `role` defines the prompt type and `content` defines the instruction

- `system` prompt: typically hidden from users and used to guide the model before user interaction
- `user` prompt: typically the question you send to a chatbot

In RAG systems, we can use these two prompts to our advantage

In [ ]:
prompt_dict = [
    {
        'role': 'system',
        'content': 'You are a renowned poet',
    },
    {
        'role': 'user',
        'content': 'Why is the sky blue?',
    }
]

In [ ]:
print(prompt_dict)

### Defining Prompt with LangChain

LangChain provides helper classes to define these roles in a more user-friendly way. Note that `user` and `human` are equivalent roles.

In [ ]:
sysmsg = SystemMessage(content="You are a renowned poet")
humanmsg = HumanMessage(content="Why is the sky blue?")

In [ ]:
lc_message = [sysmsg, humanmsg]
print(lc_message)

`prompt_dict` and `lc_message` are exactly the same when you invoke an LLM.

### Prompt Template

You've seen how to create simple prompts, but what if you want to update a prompt with new information, for example from a database. You don't need to redefine the full prompt each time, you can use a prompt template.

In [ ]:
template = ChatPromptTemplate(
    [
        ("system", "You are a renowned {job}"),
        ("human", "{question}"),
    ]
)

In [ ]:
prompt_value = template.invoke(
    {
        "job": "poet",
        "question": "Why is the sky blue?",
    }
)
prompt_value

### Prompt Template for RAG

The previous example is very simple, but for RAG systems we can design a better prompt.

Let's start by defining a `user_template`, where we provide instructions and keep `{context}` and `{question}` as variables that will be filled when we invoke the template

In [ ]:
user_template = """

Use the following pieces of context to answer the question at the end.

{context}

Question: {question}

Helpful Answer:
"""

Now, using `ChatPromptTemplate` we define the template we will pass to the LLM

In [ ]:
template = ChatPromptTemplate(
    [
        ("system", "You are a helpful assistant that answers the user questions based on the provided context. If the provided context does not contain the answer, say you don't know."),
        ("human", user_template),
    ]
)

Let's define some context and a question

In [ ]:
context = """
The daytime sky appears blue because air molecules scatter shorter wavelengths of sunlight more than longer ones (redder light).
The night sky appears to be a mostly dark surface or region spangled with stars.
"""
question = "Why is the sky blue?"

Finally, by invoking the template these variables are expanded automatically, and we are ready to send the prompt to the LLM

In [ ]:
prompt_value = template.invoke(
    {
        "context": context,
        "question": question,
    }
)
prompt_value

## Exercises for the Reader

- Instantiate an LLM model and invoke it using the different prompt formats defined in this notebook

## Conclusions

In this notebook, we explored different ways to define prompts used to invoke an LLM. We also explored how prompt templates can be expanded with runtime values before sending them to the model.

---

[AMD University Program](https://www.amd.com/aup)

Copyright (C) 2026 Advanced Micro Devices, Inc. All rights reserved. Portions of this file consist of AI-generated content.

SPDX-License-Identifier: MIT